# AMD MI300X — GPU Inference Setup & Benchmarking

This notebook:
1. Verifies ROCm and GPU access
2. Starts the vLLM server
3. Runs a throughput benchmark
4. Demonstrates batch inference for all supply chain entities

Run this **before** `01_pipeline_demo.ipynb` on the AMD environment.

## 1. GPU Health Check

In [ ]:
import subprocess

# ROCm GPU info
result = subprocess.run(['rocm-smi', '--showproductname', '--showmeminfo', 'vram'],
                        capture_output=True, text=True)
print(result.stdout or 'rocm-smi not available')

# PyTorch GPU check
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA/ROCm available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    print(f'Total VRAM: {props.total_memory / 1e9:.1f} GB')
    print(f'Used  VRAM: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB')

## 2. Start vLLM Server (run in terminal, not here)

In [ ]:
# Print the command to run in a separate terminal
MODEL_PATH = './models/Qwen2.5-14B-Instruct-GPTQ-Int4'

cmd = f"""
python -m vllm.entrypoints.openai.api_server \\
    --model {MODEL_PATH} \\
    --dtype float16 \\
    --max-model-len 8192 \\
    --gpu-memory-utilization 0.90 \\
    --host 0.0.0.0 --port 8000
"""
print('Run this in a separate terminal:')
print(cmd)

## 3. Wait for vLLM to be Ready

In [ ]:
import time, httpx

VLLM_URL = 'http://localhost:8000'

print('Waiting for vLLM server...', end='', flush=True)
for _ in range(60):   # Wait up to 5 minutes
    try:
        r = httpx.get(f'{VLLM_URL}/health', timeout=3)
        if r.status_code == 200:
            print(' Ready!')
            break
    except Exception:
        print('.', end='', flush=True)
        time.sleep(5)
else:
    print(' Timeout — check if vLLM started correctly')

## 4. Throughput Benchmark

In [ ]:
import time, asyncio
from openai import AsyncOpenAI

client = AsyncOpenAI(api_key='EMPTY', base_url=f'{VLLM_URL}/v1')

# List available models
models = await client.models.list()
print('Models:', [m.id for m in models.data])

MODEL_NAME = models.data[0].id

# Single-prompt latency
prompt = 'Summarise the key vendor risks for a semiconductor supplier headquartered in Taiwan.'

t0 = time.time()
resp = await client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{'role': 'user', 'content': prompt}],
    max_tokens=200, temperature=0.1,
)
t1 = time.time()

tokens_out = resp.usage.completion_tokens
elapsed    = t1 - t0
print(f'Single prompt latency: {elapsed:.2f}s')
print(f'Output tokens: {tokens_out}')
print(f'Throughput: {tokens_out/elapsed:.1f} tokens/sec')
print(f'\nResponse preview: {resp.choices[0].message.content[:200]}...')

## 5. Batch Inference Benchmark (simulating full pipeline)

In [ ]:
import asyncio, time

# Simulate the batch of entity narratives the risk_agent generates
ENTITIES = [
    'TSMC (Taiwan, semiconductor fabrication, importance: 10/10)',
    'Foxconn (Taiwan, contract manufacturing, importance: 9/10)',
    'Samsung (Korea, memory/display, importance: 8/10)',
    'Broadcom (USA, wireless chips, importance: 8/10)',
    'ASML (Netherlands, lithography equipment, importance: 9/10)',
    'Applied Materials (USA, semiconductor equipment, importance: 8/10)',
    'Shin-Etsu Chemical (Japan, silicon wafers, importance: 7/10)',
    'ARM Holdings (UK, IP licensing, importance: 9/10)',
    'Corning (USA, specialty glass, importance: 7/10)',
    'Alphabet (USA, search partner, importance: 9/10)',
]

prompts = [
    f'Write a 3-sentence vendor risk assessment for: {e}. Focus on operational and geopolitical risks.'
    for e in ENTITIES
]

async def batch_generate(prompts, concurrency=5):
    sem = asyncio.Semaphore(concurrency)
    async def one(p):
        async with sem:
            r = await client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{'role': 'user', 'content': p}],
                max_tokens=150, temperature=0.3,
            )
            return r.choices[0].message.content, r.usage.completion_tokens
    return await asyncio.gather(*[one(p) for p in prompts])

t0      = time.time()
results = await batch_generate(prompts)
t1      = time.time()

total_tokens = sum(r[1] for r in results)
elapsed      = t1 - t0
print(f'Batch size:       {len(prompts)} prompts')
print(f'Total time:       {elapsed:.2f}s')
print(f'Total tokens out: {total_tokens}')
print(f'Throughput:       {total_tokens/elapsed:.1f} tokens/sec')
print(f'Avg per entity:   {elapsed/len(prompts):.2f}s')
print(f'\nSample output for TSMC:')
print(results[0][0])

## 6. Run Full Pipeline with vLLM

In [ ]:
import sys
sys.path.insert(0, '..')

from src.pipeline.workflow import run_pipeline
from pathlib import Path

state = await run_pipeline(
    target_company='Apple Inc',
    target_ticker='AAPL',
    llm_backend='vllm',
)

print(f'Entities:  {len(state.entities)}')
print(f'Scored:    {len(state.risk_scores)}')
print(f'Alerts:    {len(state.alerts)}')
print(f'Errors:    {len(state.errors)}')

# Save dashboard
out = Path('../data/outputs/apple_vllm_dashboard.html')
out.write_text(state.dashboard_html, encoding='utf-8')
print(f'Dashboard: {out}')